# 01 — Bố trí thí nghiệm cho công bằng: các lỗi đo đạc đã gặp

Notebook 00 xây công cụ thống kê; notebook này dùng chúng để trả lời một câu khó hơn:
**làm sao biết con số mình đo được là thật?** Câu trả lời đi qua các lỗi đo đạc thật đã xảy ra
trong dự án — bốn lỗi ghi trong README, và hai lỗi nữa phát hiện ra khi viết bộ notebook này.

Điểm chung của cả sáu: chúng đều **làm kết quả trông đẹp hơn hoặc chắc chắn hơn thực tế**. Đó là
đặc điểm nguy hiểm nhất của lỗi đo — nó thưởng cho người không kiểm tra, nên không có động lực tự
nhiên để tìm ra.

Nền tảng cần có: notebook 00 (McNemar, lực kiểm định, khoảng tin cậy). Notebook này chỉ đọc các
file kết quả trong `results/`, không chạy lại mô hình.

| Lỗi | Loại | Hậu quả nếu không phát hiện |
|---|---|---|
| 1 | Vùng bấm giờ không công bằng | tăng tốc bị thổi phồng gấp ba |
| 2 | Mỗi vòng đo một nhóm mẫu khác | độ khó của mẫu bị đọc thành nhiễu của máy |
| 3 | Kết luận "không khác biệt" khi thiếu lực | kết luận sai chiều |
| 4 | Giá trị mặc định lặng lẽ đổi thí nghiệm | suýt kết luận sai về lượng tử hoá |
| 5 | Đếm vòng lặp như mẫu mới | khoảng tin cậy hẹp hơn thực tế |
| 6 | Mượn ngưỡng nhiễu của một mô hình khác | ngưỡng tuyên bố cải thiện sai lệch |

In [1]:
import json, math, statistics, sys
from collections import defaultdict
from pathlib import Path

import numpy as np

sys.path.insert(0, "..")                      # main-branch code lives one level up
from bench.metrics import paired_accuracy, wilson_interval

RESULTS = Path("../results")

def load(name):
    return json.loads((RESULTS / name).read_text())

print("result files:", ", ".join(p.name for p in sorted(RESULTS.glob("*.json"))))

result files: breakdown.json, gate0_latency.json, gate1_smolvlm2b.json, gate2_confirm.json, gate2_edge_sweep.json, gate2_sweep.json, gate3_quant.json, gate4_docker_1536.json, gate4_docker_768.json, gate4_serving_1536.json, gate4_serving_768.json, gate5_nf4.json, gate5_nf4_300.json, gate6_prompt.json, preprocess_cost.json, prune_smoke.json, prune_smoke2.json, sanity_docvqa.json, sanity_docvqa_en.json, smoke.json, sweep.json


## Lỗi 1 — vùng bấm giờ không công bằng

Kỹ thuật cắt token ảnh cần lấy token ảnh ra **trước** khi đưa vào mô hình ngôn ngữ, nên đường tối
ưu gọi bộ mã hoá thị giác sớm hơn. Trong phiên bản đầu, lời gọi đó nằm **ngoài** vùng bấm giờ.
Kết quả: đường gốc được tính cả thời gian mã hoá thị giác, đường tối ưu thì không — một bên còn
gánh, một bên đã đặt gánh xuống.

Hai file dưới đây là cùng một thí nghiệm, trước và sau khi sửa:

In [2]:
before, after = load("prune_smoke.json"), load("prune_smoke2.json")
for name, r in [("BEFORE the fix", before), ("AFTER the fix", after)]:
    print(name)
    for k, v in r["paired_speedup_vs_baseline"].items():
        print(f"   {k.split('(')[0]:<12} {v['median_speedup']:.2f}x")

BEFORE the fix
   keep0.5      3.47x
   keep0.25     4.36x
AFTER the fix
   keep0.5      1.12x
   keep0.25     1.22x


Cùng kỹ thuật, cùng mô hình, cùng máy: **3,47× thành 1,12×**. Notebook 02 (định luật Amdahl)
cho biết trần lý thuyết của kỹ thuật này là khoảng 1,42× — nên một con số 3,47× lẽ ra phải bị
nghi ngờ ngay: nó **vượt trần vật lý**.

**Quy tắc 1:** vùng bấm giờ phải bao trọn toàn bộ đường đi mà người dùng thật phải trả, ở **mọi**
cấu hình được so sánh. Và: **đối chiếu kết quả với một giới hạn lý thuyết** — con số vượt trần là
dấu hiệu lỗi đo, không phải đột phá.

## Lỗi 2 — mỗi vòng đo dùng một nhóm mẫu khác

Ban đầu mỗi vòng đo một nhóm câu hỏi khác nhau. Nghe hợp lý, nhưng ảnh ChartQA có kích thước rất
khác nhau, nên thời gian xử lý mỗi câu dao động mạnh **vì bản thân câu hỏi**, không vì máy. Triệu
chứng lúc đó: IQR bằng 86,5% trung vị, và "độ trôi" −44% — máy càng chạy càng nhanh, điều vô lý.

Tách hai nguồn dao động trên dữ liệu thật (`gate2_confirm`, mỗi câu đo hai vòng):
- **Dao động giữa các câu hỏi**: độ phân tán của thời gian qua 300 câu khác nhau.
- **Nhiễu của máy**: độ phân tán của tỉ số giữa hai vòng của **cùng** một câu (notebook 02, mục 3.1).

In [3]:
confirm = load("gate2_confirm.json")
by = defaultdict(dict)
for r in confirm["records"]:
    if r["config"] == "baseline":
        by[r["sample_id"]][r["round_idx"]] = r["generate_ms"]
first_round = np.array([v[0] for v in by.values()])
ratio = np.array([v[1] / v[0] for v in by.values()])

def iqr_pct(x):
    return 100 * (np.percentile(x, 75) - np.percentile(x, 25)) / np.median(x)

print(f"spread across questions (round 1)     : IQR {iqr_pct(first_round):5.1f}% of median")
print(f"machine noise (same question, 2 rounds): IQR {iqr_pct(ratio):5.1f}% of median ratio")

spread across questions (round 1)     : IQR  13.9% of median
machine noise (same question, 2 rounds): IQR   5.7% of median ratio


Dao động do câu hỏi lớn gấp khoảng 2,5 lần nhiễu của máy, ngay cả ở cấu hình gốc nơi mọi ảnh đều
thành 13 ô. Nếu hai cấu hình được đo trên hai **nhóm câu khác nhau**, chênh lệch giữa hai nhóm có thể
lấn át hiệu ứng thật. Một ví dụ đồ chơi cho thấy nó có thể
đảo ngược kết luận:

In [4]:
cost = {"small image": 100.0, "medium image": 400.0, "large image": 1000.0}
optimised = {k: v / 2 for k, v in cost.items()}                     # truly 2x faster everywhere
group_a = statistics.median([cost["small image"], cost["medium image"]])
group_b = statistics.median([optimised["medium image"], optimised["large image"]])
print(f"different questions per config : {group_a / group_b:.2f}x  (wrong)")
print(f"paired, per question           : {statistics.median(cost[k] / optimised[k] for k in cost):.2f}x  (right)")

different questions per config : 0.71x  (wrong)
paired, per question           : 2.00x  (right)


So hai nhóm khác nhau cho **0,71×** — kết luận rằng kỹ thuật làm mọi thứ **chậm đi** — trong khi
thật ra nó nhanh đúng **2×**. Đây là cùng nguyên lý với McNemar ở notebook 00: **so theo cặp trên
cùng câu hỏi** làm độ khó của câu hỏi tự triệt tiêu.

**Quy tắc 2:** các vòng đo là **bản lặp trên cùng tập câu hỏi**, và mọi so sánh làm theo cặp trên
từng câu, rồi mới tổng hợp.

## Lỗi 3 — kết luận "không khác biệt" khi thiếu lực

Với 100 câu, cạnh 768 kém bản gốc 6 điểm, p = 0,18 — "không phân biệt được". Rất dễ đọc thành
"giảm độ phân giải không làm mất chất lượng". Chạy lại với 300 câu:

In [5]:
for name, f in [("100 samples", "gate2_edge_sweep.json"), ("300 samples", "gate2_confirm.json")]:
    r = load(f)
    pa = paired_accuracy(r["records"], "baseline", "edge768(edge=768)")
    a = 100 * r["configs"]["baseline"]["accuracy"]; b = 100 * r["configs"]["edge768(edge=768)"]["accuracy"]
    print(f"{name}: {a:.1f}% -> {b:.1f}% ({b - a:+.1f} pts) | discordant "
          f"{pa['only_a_correct']}+{pa['only_b_correct']} = {pa['only_a_correct'] + pa['only_b_correct']} | p = {pa['p_value']:.4f}")

100 samples: 69.0% -> 63.0% (-6.0 pts) | discordant 10+4 = 14 | p = 0.1796
300 samples: 64.7% -> 57.3% (-7.3 pts) | discordant 35+13 = 48 | p = 0.0021


Hiệu ứng gần như không đổi (6,0 rồi 7,3 điểm), nhưng số cặp bất đồng tăng từ 14 lên 48, và kết
luận đảo ngược. Notebook 00 (mục 6) tính được nguyên nhân: với hiệu ứng cỡ này, **lực kiểm định ở
100 câu chỉ khoảng 0,37** — cứ ba lần làm thí nghiệm thì khoảng hai lần ra "không có ý nghĩa", dù
hiệu ứng có thật. p = 0,18 là kết cục *có khả năng nhất*, không phải một sự cố hiếm.

**Quy tắc 3:** "p lớn" nghĩa là *chưa đủ bằng chứng*, không phải *không có khác biệt*. Trước khi
kết luận "không khác biệt", hãy báo **khoảng tin cậy của hiệu số** (nó cho biết hiệu ứng lớn nhất
còn tương thích với dữ liệu), và ước lượng lực từ một lần chạy thử.

## Lỗi 4 — giá trị mặc định lặng lẽ đổi thí nghiệm

Một lần đo quên truyền `--model`, nên bộ đo dùng mặc định lúc đó là mô hình **256 triệu tham số**
thay vì bản **2,2 tỉ** dùng xuyên suốt. Kết quả trông như một thảm hoạ: độ chính xác tụt từ 64%
xuống 23%, số token ảnh đổi từ 1.053 thành 640. Suýt nữa đã kết luận rằng lượng tử hoá 4 bit phá
hỏng mô hình.

Dấu hiệu lẽ ra phải thấy ngay: **số token ảnh đổi**. Số token ảnh là hàm của bộ tiền xử lý và kiến
trúc (notebook 03), không phụ thuộc kiểu số. Một đại lượng lẽ ra **bất biến** mà lại đổi, tức là
đang đo một thứ khác với thứ mình nghĩ.

In [6]:
print(f"{'file':<24} {'model':<18} {'dtype':<6} {'image tokens':>12}")
for f in ["gate2_edge_sweep.json", "gate2_confirm.json", "gate5_nf4.json", "gate5_nf4_300.json", "gate6_prompt.json"]:
    r = load(f); c = r["configs"]["baseline"]
    print(f"{f:<24} {r['model'].split('/')[-1]:<18} {r.get('quant', 'bf16'):<6} {c['image_tokens_median']:>12.0f}")

file                     model              dtype  image tokens
gate2_edge_sweep.json    SmolVLM-Instruct   bf16           1053
gate2_confirm.json       SmolVLM-Instruct   bf16           1053
gate5_nf4.json           SmolVLM-Instruct   nf4            1053
gate5_nf4_300.json       SmolVLM-Instruct   nf4            1053
gate6_prompt.json        SmolVLM-Instruct   bf16           1053


Mọi lần chạy bf16 và nf4 của cùng mô hình đều cho 1.053 token ảnh — đúng như phải thế. Sau lỗi này,
mặc định của `--model` được đổi sang bản 2.2B, và tên mô hình cùng số token ảnh được ghi vào mọi
file kết quả.

**Quy tắc 4:** ghi lại **đại lượng bất biến** cùng mỗi phép đo — tên mô hình, số token ảnh — và kiểm
tra chúng **trước** khi đọc kết quả.

## Lỗi 5 — đếm vòng lặp như mẫu mới

*(Phát hiện khi viết notebook 00.)* `bench/harness.py` tính khoảng tin cậy Wilson của độ chính xác
với $n$ là **số bản ghi** — số câu × số vòng. Nhưng mô hình giải mã tất định: câu nào sai ở vòng 1
thì sai ở mọi vòng. Các vòng lặp **không** mang thêm thông tin về độ chính xác.

In [7]:
sweep = load("gate2_sweep.json")
per = defaultdict(set)
for r in sweep["records"]:
    if r["config"] == "baseline":
        per[r["sample_id"]].add(r["correct"])
print(f"baseline, 40 questions x 3 rounds: {sum(len(v) == 1 for v in per.values())}/40 questions "
      f"give the same verdict in every round")
k = sum(next(iter(v)) for v in per.values())
stored = sweep["configs"]["baseline"]["accuracy_ci95"]
lo, hi = wilson_interval(k, 40)
print(f"stored CI (n = 120 records) : [{100 * stored[0]:.1f}, {100 * stored[1]:.1f}]  width {100 * (stored[1] - stored[0]):.1f} pts")
print(f"correct CI (n = 40 questions): [{100 * lo:.1f}, {100 * hi:.1f}]  width {100 * (hi - lo):.1f} pts")

baseline, 40 questions x 3 rounds: 40/40 questions give the same verdict in every round
stored CI (n = 120 records) : [46.1, 63.6]  width 17.5 pts
correct CI (n = 40 questions): [39.8, 69.3]  width 29.5 pts


Khoảng tin cậy lưu trong file hẹp hơn khoảng đúng khoảng $\sqrt 3$ lần. Các thanh sai số trên biểu
đồ trong README vì vậy **trông chắc chắn hơn thực tế**. Không kết luận chính nào đổi (các kết luận
chính dựa trên McNemar theo cặp, vốn tính đúng trên từng câu), nhưng biểu đồ cần sửa.

**Quy tắc 5:** $n$ là **số quan sát độc lập**, không phải số dòng trong file. Lặp lại một phép đo tất
định giúp đo **thời gian** chính xác hơn, nhưng không thêm thông tin về **độ chính xác**.

## Lỗi 6 — mượn ngưỡng nhiễu của một mô hình khác

*(Phát hiện khi viết notebook 02.)* Quy tắc "chỉ tuyên bố cải thiện vượt ba lần nhiễu" dùng ngưỡng
25,5% = 3 × 8,5%. Con số 8,5% đo ở cổng 0 — trên **mô hình 256M**, trước khi dự án chuyển sang bản
2.2B. Nhiễu của mô hình 2.2B, đo từ chính các vòng lặp của dự án:

In [8]:
log_ratio = np.log(ratio)
print(f"2.2B model noise per measurement: {100 * log_ratio.std(ddof=1) / math.sqrt(2):.1f}%  "
      f"-> 3x threshold {300 * log_ratio.std(ddof=1) / math.sqrt(2):.0f}%")
print(f"gate 0 file today: model {load('gate0_latency.json')['model'].split('/')[-1]}, "
      f"CV {load('gate0_latency.json')['generate_ms']['cv_pct']:.2f}%  (the 8.5% run was later overwritten)")

2.2B model noise per measurement: 3.2%  -> 3x threshold 10%
gate 0 file today: model SmolVLM-256M-Instruct, CV 7.75%  (the 8.5% run was later overwritten)


Nhiễu thật của mô hình 2.2B chỉ khoảng 3%, nên ngưỡng đúng khoảng 9–10%, không phải 25,5%. Lỗi này
đi theo chiều **thận trọng**: không kết luận nào của dự án bị ảnh hưởng (1,93× vượt xa cả hai
ngưỡng), nhưng một cải thiện thật cỡ 12–20% sẽ bị gạt bỏ oan. Nó cũng là một biến thể của lỗi 4: một
con số đo trong **điều kiện khác** được mang sang mà không đo lại. File kết quả gốc của lần đo 8,5%
còn bị ghi đè sau đó, nên con số trong README không tái lập được từ các file hiện có.

**Quy tắc 6:** mọi hằng số dùng để ra quyết định (ngưỡng nhiễu, mức nền) phải được đo **trong đúng
điều kiện** của thí nghiệm nó phục vụ, và file gốc của nó phải được giữ lại.

## Danh sách kiểm tra

Sáu quy tắc đo trong `bench/harness.py` (notebook 02, bảng cuối) chặn các lỗi 1–4. Hai lỗi mới thêm
hai mục. Trước khi tin một kết quả:

1. Vùng bấm giờ có bao trọn đường đi thật, ở **mọi** cấu hình? Con số có vượt trần lý thuyết không?
2. Hai cấu hình có được so **theo cặp**, trên cùng câu hỏi?
3. "Không khác biệt" có đi kèm khoảng tin cậy và ước lượng lực?
4. Các đại lượng bất biến (mô hình, số token ảnh) có đúng như dự kiến?
5. $n$ có phải là số quan sát **độc lập**?
6. Các ngưỡng và hằng số có được đo trong **đúng điều kiện** hiện tại?

Và một quy tắc không cài được vào code, chỉ nằm ở thói quen: **khi một kết quả đẹp bất ngờ, nghi
ngờ phép đo trước khi mừng.**